<a href="https://colab.research.google.com/github/mrdantzig/nlp-bible-code/blob/master/15%EC%9E%A5_Text%20summarization/%5B15-2%5D%20%EB%B9%84%EC%A7%80%EB%8F%84%20%ED%95%99%EC%8A%B5%EC%9D%84%20%EC%9D%B4%EC%9A%A9%ED%95%9C%20%EC%9D%8C%EC%8B%9D%20%EB%A6%AC%EB%B7%B0%20%EC%B6%94%EC%B6%9C%20%EC%9A%94%EC%95%BD_%EC%88%98%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [15-2] 비지도 학습을 이용한 음식 리뷰 추출 요약 (수정본)

자연어처리 바이블 15장 실습 [15-2]를 최신 Colab에서 그대로 실행되도록 고친 노트북입니다. 정답 요약 없이 **K-means 군집화**로 리뷰마다 대표 문장을 골라 요약합니다.

**원본 대비 수정 사항**
1. **데이터**: 300MB `Reviews.csv`를 브라우저로 올리면 느리고 중간에 끊기기 쉬워, `kagglehub`로 내려받습니다. 실패하면 원본처럼 직접 업로드합니다.
2. **NLTK**: 최신 NLTK의 `sent_tokenize`에는 `punkt_tab`이 필요합니다.
3. **Keras Tokenizer 셀 삭제**: 요약에 쓰이지 않습니다. Keras 3에서는 `keras.preprocessing.text`가 없어 오류도 납니다.
4. **GloVe**: 1.4GB 압축 파일을 내려받아 올리는 대신 `gensim.downloader`로 같은 벡터(`glove-twitter-25`, 약 105MB)를 받습니다. 단어 수도 원본과 같은 1,193,514개입니다.
5. **`.get()` → `in`**: gensim의 `KeyedVectors`에는 `.get()`이 없어 `k in emb_index`로 확인합니다.
6. **문장 번호 버그 수정**: 원본은 짧은 문장을 건너뛰면 군집 대표와 다른 문장을 요약에 넣었습니다. 남은 문장(`kept`)을 함께 반환해 바로잡았습니다.
7. **진행 표시**: 5,000줄 출력 대신 500개마다 진행 상황을 표시합니다.
8. **HTML 태그 정리**: 리뷰 본문의 `<br />` 태그를 문장 분리 전에 공백으로 바꿉니다.

## 0. 라이브러리 준비

In [ ]:
!pip install -q gensim kagglehub

In [ ]:
import os
import io
import re
import string

import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')   # 수정: 최신 NLTK의 sent_tokenize에 필요

## 1. 데이터 불러오기

Kaggle **Amazon Fine Food Reviews**(https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews)의 `Reviews.csv`를 사용합니다. 전체 데이터는 568,454개 리뷰, 10개 열입니다.

In [ ]:
# 수정: kagglehub로 내려받고, 실패하면 원본처럼 직접 업로드
try:
    import kagglehub
    path = kagglehub.dataset_download('snap/amazon-fine-food-reviews')
    df = pd.read_csv(os.path.join(path, 'Reviews.csv'))
except Exception as e:
    print('kagglehub 다운로드 실패 → Reviews.csv를 직접 업로드해 주세요.', e)
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(io.BytesIO(uploaded['Reviews.csv']))

print(df.shape)   # 전체 데이터라면 (568454, 10)
if len(df) < 568454:
    print('※ 전체 데이터보다 적습니다. 업로드가 중간에 끊기지 않았는지 확인해 주세요.')

In [ ]:
df.head(3)

## 2. 리뷰를 문장 단위로 나누기

리뷰 원문에는 웹 페이지의 줄바꿈 태그 `<br />`가 섞여 있습니다. 태그가 마침표 바로 뒤에 붙어 있으면 `sent_tokenize`가 문장 경계를 찾지 못해 `).<br />The little ...`처럼 앞뒤 문장이 붙은 채로 요약에 나옵니다. 문장을 나누기 전에 태그를 공백으로 바꿔 줍니다.

In [ ]:
# 추가: HTML 줄바꿈 태그(<br />, <br/>, <br>)를 공백으로 바꾸기
df['Text'] = df['Text'].str.replace(r'<br\s*/?>', ' ', regex=True)

In [ ]:
def split_sentences(reviews):
    """
    Splits the reviews into individual sentences
    """
    n_reviews = len(reviews)
    for i in range(n_reviews):
        review = reviews[i]
        sentences = sent_tokenize(review)
        for j in reversed(range(len(sentences))):
            sent = sentences[j]
            sentences[j] = sent.strip()
            if sent == '':
                sentences.pop(j)
        reviews[i] = sentences

In [ ]:
rev_list = list(df['Text'])
split_sentences(rev_list)
# Adding split reviews in the data frame
df['sent_tokens'] = rev_list
# Calculating length of sentences in each review
df['length_of_rv'] = df['sent_tokens'].map(lambda x: len(x))

In [ ]:
choice_length = 5
df = df[df['length_of_rv'] > choice_length]   # 6문장 이상인 리뷰만 사용
df.shape   # 전체 데이터라면 (172765, 12)

> 원본에는 이 다음에 Keras `Tokenizer`/`pad_sequences`로 `X_t`를 만드는 셀이 있었습니다. `X_t`는 요약에 쓰이지 않고, Keras 3에서는 `keras.preprocessing.text`가 없어 오류가 나므로 뺐습니다.

## 3. 단어 임베딩 불러오기 (GloVe Twitter 25차원)

In [ ]:
import gensim.downloader as api

# 수정: GloVe 파일을 직접 올리는 대신 gensim downloader로 같은 벡터를 받기(약 105MB, 1~2분)
emb_index = api.load('glove-twitter-25')
print('Loaded %s word vectors.' % len(emb_index))   # 1193514

## 4. 문장 임베딩 = 단어 벡터의 평균

In [ ]:
def calculate_sentence_embedding(wordList):
    """
    문장에 포함된 단어 벡터들의 평균으로 문장 벡터를 만든다.
    """
    # 수정: KeyedVectors에는 .get()이 없으므로 in으로 확인
    emb_li = [emb_index[k] for k in wordList if k in emb_index]
    if len(emb_li) == 0:
        return None
    return np.mean(np.array(emb_li), axis=0)


def get_sent_embedding(mylist):
    """
    리뷰의 각 문장을 벡터로 바꾼다. 단어 2개 이하인 문장(예: "!", ":)")은 제외한다.
    수정: 벡터로 바꾼 문장 목록(kept)도 함께 돌려줘서 문장 번호가 어긋나지 않게 한다.
    """
    sent_emb, kept = [], []
    for sent in mylist:
        wL = re.sub(r"[^\w]", " ", sent.lower()).split()
        wL = [k for k in wL if k not in string.punctuation]
        if len(wL) <= 2:
            continue
        emb = calculate_sentence_embedding(wL)
        if emb is None:
            continue
        sent_emb.append(emb)
        kept.append(sent)
    return np.array(sent_emb), kept

## 5. K-means로 리뷰 요약하기

- 군집 수 k = ⌈√문장 수⌉
- 각 군집 중심(centroid)에 가장 가까운 문장을 대표 문장으로 고른다.
- 군집 안 문장들의 평균 위치 순으로 대표 문장을 이어 붙인다.

※ 5,000개 리뷰를 요약하는 데 몇 분 걸립니다. 처음에는 `how_many_summaries`를 20 정도로 줄여 시험해 보세요.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min

## Calculating summaries for first 5000 reviews.
how_many_summaries = min(5000, len(df))
summary = [None] * how_many_summaries
for rv in range(how_many_summaries):
    review = df['sent_tokens'].iloc[rv]
    enc_email, kept = get_sent_embedding(review)
    if len(enc_email) > 0:
        n_clusters = int(np.ceil(len(enc_email) ** 0.5))
        kmeans = KMeans(n_clusters=n_clusters, random_state=0)
        kmeans = kmeans.fit(enc_email)
        avg = []
        for j in range(n_clusters):
            idx = np.where(kmeans.labels_ == j)[0]
            avg.append(np.mean(idx))
        closest, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, enc_email)
        ordering = sorted(range(n_clusters), key=lambda k: avg[k])
        # 수정: review가 아니라 kept에서 문장을 꺼낸다
        summary[rv] = ' '.join([kept[closest[idx]] for idx in ordering])
    else:
        print('This is not a valid review: #', rv)
    if (rv + 1) % 500 == 0 or rv + 1 == how_many_summaries:
        print('Done for review # =', rv + 1, '/', how_many_summaries)

## 6. 결과 저장과 확인

In [ ]:
df_5000 = df.iloc[:how_many_summaries].copy()
df_5000['PredictedSummary'] = summary
df_5000[['Text', 'PredictedSummary']].to_csv('top_5000_summary.csv')

In [ ]:
df_result = pd.read_csv('top_5000_summary.csv')
df_result.head(3)

## 7. (추가) 원문과 요약 나란히 보기

In [ ]:
for i in range(3):
    print(f'[원문 {i+1}]', df_5000['Text'].iloc[i])
    print(f'[요약 {i+1}]', df_5000['PredictedSummary'].iloc[i])
    print('-' * 80)